# GENAI Pioneer Banking Intelligence Session Lab**

Cell 1: Environment Setup & Installations
This cell installs all necessary external dependencies.

The enterprise document AI workflow is designed to process bank statements, starting with simulated OCR text data. Here's a summary of its architecture:

Data Ingestion & Extraction: A SimulatedOCRProvider provides the raw text. This text is then processed by RegEx-based functions (extract_header_entities and extract_transactions) to parse out key header information (like bank name, account holder, PAN, IFSC) and individual transaction details.

Data Validation: Extracted data is validated against Pydantic models (Transaction and BankDocument). These models enforce strict schema rules and data type checks, including specific format validations for fields like PAN, IFSC, and account numbers, ensuring data integrity before further processing.

Enrichment - Categorization & Sentiment Analysis: Each transaction's description is enriched by two rule-based functions:

categorize_transaction: Assigns a category (e.g., 'Income', 'Food & Dining', 'Transfer') based on keywords in the description.
analyze_sentiment: Determines a sentiment (e.g., 'Positive', 'Negative', 'Neutral') for the transaction, also based on keywords.
Business Intelligence & Risk Assessment: Two core functions provide risk insights:

generate_risk_flags: Identifies potential anomalies or high-risk activities (e.g., high-value debits/credits, international transfers, unknown beneficiaries) within the transactions.
ai_summary_pipeline: Consolidates the risk flags to generate an overall risk score and a recommended action (e.g., 'MANUAL_REVIEW_REQUIRED', 'LOW_RISK') for the entire document.
Orchestration & Auditing: The enterprise_document_ai_workflow function orchestrates all these steps. It maintains detailed audit_logs for transparency and debugging, providing real-time progress updates for each stage.

API Deployment: The entire workflow is exposed as a REST API endpoint (/process-document) using FastAPI, running in a background thread. This allows for easy integration with external systems, modularity, real-time processing, and standardized interaction through automatically generated API documentation.

In [ ]:
# Install required libraries for data structures, API simulation, and background execution
!pip install -q pydantic fastapi uvicorn nest_asyncio pandas requests

Cell 2: Core Architecture Imports
This cell imports the core libraries needed for standardizing data schemas, regular expressions, and unique identifiers.

In [ ]:
import re
import json
import uuid
import threading
import pandas as pd
from datetime import datetime, timezone
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field, ValidationError, field_validator
import nest_asyncio
import uvicorn
import requests
from fastapi import FastAPI, HTTPException

print("✅ Imports completed successfully.")

✅ Imports completed successfully.


Cell 3: Simulated OCR Layer & Source Text
This cell abstracts the data ingestion layer and sets up a fixed mock bank statement to guarantee reliable execution without API keys.

In [ ]:
# Install required libraries for OCR
!pip install -q pytesseract pdf2image
!apt-get install -y poppler-utils # For pdf2image

import io
from PIL import Image
import pytesseract
from pdf2image import convert_from_path
# from google.colab import files # No longer needed for direct file path
import os # Still needed for path manipulation and file existence checks
import uuid # Not strictly needed if not creating temp files, but can keep for consistency or future use

# Configure Tesseract path if necessary (Colab usually has it pre-installed or easily available)
# pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract' # Example path

class OCRProvider:
    def extract_text(self, document_path: str) -> str: # Renamed from document_path_placeholder to document_path
        raise NotImplementedError

class RealOCRProvider(OCRProvider):
    def extract_text(self, document_path: str) -> str:
        """
        Extracts text from a given file path (image or PDF) using Tesseract OCR.
        Assumes the file already exists at the specified document_path.
        """
        if not os.path.exists(document_path):
            raise FileNotFoundError(f"Document not found at: {document_path}")

        extracted_text = ""
        file_extension = os.path.splitext(document_path)[1].lower()

        if file_extension in ('.png', '.jpg', '.jpeg', '.tiff', '.bmp', '.gif'):
            try:
                img = Image.open(document_path)
                extracted_text = pytesseract.image_to_string(img)
            except Exception as e:
                print(f"Error processing image {document_path}: {e}")
        elif file_extension == '.pdf':
            try:
                # convert_from_path requires poppler-utils, installed above
                images = convert_from_path(document_path)
                for i, image in enumerate(images):
                    extracted_text += pytesseract.image_to_string(image) + "\n"
            except Exception as e:
                print(f"Error processing PDF {document_path}: {e}")
        else:
            raise ValueError(f"Unsupported file type: {file_extension}. Please provide an image or PDF.")

        if not extracted_text.strip():
            print(f"Warning: No text extracted from {document_path}. It might be a scanned document with no recognizable text or an issue with OCR.")

        print(f"Successfully extracted text from {document_path}.")
        return extracted_text.strip()

# Initialize the real OCR engine.
# Now, every time ocr_engine.extract_text is called, it will process the file at the provided path.
ocr_engine = RealOCRProvider()
print("✅ Real OCR Abstraction Layer initialized. Ready to process documents from specified paths.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
✅ Real OCR Abstraction Layer initialized. Ready to process documents from specified paths.


### Verification of Extraction Functions
Let's run the `extract_header_entities` and `extract_transactions` functions on the `sample_ocr_text` to inspect their direct output.

In [ ]:
print("Processing bank statement using the real OCR engine...")
ocr_text = ocr_engine.extract_text("/content/BANK STATEMENT 2.pdf")

extracted_header = extract_header_entities(ocr_text)
extracted_transactions = extract_transactions(ocr_text)

print("\n--- Extracted Header Entities ---")
for key, value in extracted_header.items():
    print(f"{key}: {value}")

print("\n--- Extracted Transactions ---")
for i, txn in enumerate(extracted_transactions):
    print(f"Transaction {i+1}: {txn}")

Processing bank statement using the real OCR engine...
Successfully extracted text from /content/BANK STATEMENT 2.pdf.

--- Extracted Header Entities ---
account_number: 81198100000007
ifsc: BARBOVJMDNG
bank_name: World Bank of Baroda
customer_name: CH JANAKI RAGHU RAMI REDDY MADHAVANAGAR KURNOOL
pan: None
mobile: None
email: None
statement_period: 01-12-2024 to 31-01-2025

--- Extracted Transactions ---
Transaction 1: {'date': '03-12-2024', 'description': 'Opening Balance', 'transaction_type': 'UNKNOWN', 'amount': 0.0, 'balance': 0.35}
Transaction 2: {'date': '05-12-2024', 'description': 'pr a TOSOTOSESSIA ASS: O/UPISSOBEBEBOS IDFA', 'transaction_type': 'CREDIT', 'amount': 1200.0, 'balance': 1200.35}
Transaction 3: {'date': '05-12-2024', 'description': 'UR SEERA UA SERIA', 'transaction_type': 'DEBIT', 'amount': 150.0, 'balance': 950.35}
Transaction 4: {'date': '10-12-2024', 'description': '(PuaT11812851.44/19°30:05/lPubharatpe. 200693587', 'transaction_type': 'DEBIT', 'amount': 10.0, 

### Reviewing the Extracted OCR Text

Below is the raw text extracted from your uploaded document. Please examine it carefully. We need to identify consistent patterns or labels for the information you want to extract (e.g., 'Account Holder', 'Statement Period', transaction lines).


In [ ]:
print(ocr_text)

print("\n--- Guidance for Adapting Regexes ---")
print("Based on the `ocr_text` above, you will need to modify the `patterns` dictionary in the `extract_header_entities` function and the `transaction_pattern` in the `extract_transactions` function, both located in **Cell 4: AI Extraction Layer (RegEx Parsing Engines)** (cell ID `tlrWKY61FgW5`).")
print("\nFor example, if 'Account Holder' is now 'Customer Name', you'd change its regex pattern. If transaction lines have a different format, the `transaction_pattern` needs adjustment.")
print("\nIf you tell me which specific pieces of information you want to extract from the `ocr_text` and their surrounding text, I can help you formulate the new regex patterns.")

bob [Bp 20 3 wt

World Bank of Baroda

feat 01-12-2024 4 31-01-2025 dw al tara faazoit

Account Statement from 01-12-2024 to 31-01-2025

 

 

waren feat / Account details

WES GTA / Account Name 20M TATA / Branch Name
CH JANAKI RAGHU RAMI REDDY MADHAVANAGAR KURNOOL
‘Mat HM / Account Number BTSTHTEE BS / IFSC Code
81198100000007 BARBOVJMDNG
lal THR / Account Type VASTSHISIN HS / MICR Code
SBA 518012008

Teh BI Vat / Customer Address al FT Taq / Branch Address

 

 

 

 

 

77140-11-C4 AYYAPPASWAMY NAGAR KURNOOL 8 7 749 BAHU PLAZA KURNOOL
KALLUR KURNOOL ANDHRA PRADESH ANDHRA PRADESH 518002 AP
KURNOOL ANDHRA PRADESH, INDIA
518003
Pa aderet waa ferret ania are star as
aka ; .
Sr.No Transaction Value Description Cheque Debit Credit Balance
Date Date Number
1 03-12-2024 Opening Balance - - 0.35
2 03-12-2024 03-12-2024 isaac ceca aeaaaad . 17,000.00 17,000.35
3 03-12-2024 03-12-2024 asa 7289/28-41:06/UP ichitreddynymavathi 17,000.00 . 0.35
4 05-12-2024 05-12-2024 pr a TOSOTOSESSIA ASS: O/U

Cell 4: AI Extraction Layer (RegEx Parsing Engines)
This block converts unstructured document blobs into distinct dictionary fragments using raw strings to avoid compilation bugs.

In [ ]:
def extract_header_entities(text: str) -> Dict[str, Any]:
    """Extracts high-level customer profile fields using strict RegEx constraints."""
    data = {}

    # Special handling for account_number and ifsc which appear on a subsequent line
    # This pattern matches 'Account Number', then anything, then 'IFSC Code' on the same line,
    # then a newline, then captures the account number and IFSC from the next line.
    account_ifsc_pattern = re.search(
        r"Account Number.*?IFSC Code\s*\n\s*(?P<account_number>\d{10,18})\s+(?P<ifsc>[A-Z]{4}[0O][A-Z0-9]{6})",
        text, re.IGNORECASE | re.DOTALL
    )
    if account_ifsc_pattern:
        data["account_number"] = account_ifsc_pattern.group("account_number").strip()
        data["ifsc"] = account_ifsc_pattern.group("ifsc").strip()
    else:
        data["account_number"] = None
        data["ifsc"] = None

    patterns = {
        "bank_name": r"World Bank of Baroda", # Specific to this document
        "customer_name": r"Account Name.*?Branch Name\s*\n\s*([^\n]+)", # Captures the line after 'Account Name ... Branch Name'
        "pan": r"PAN:\s*([A-Z]{5}[0-9]{4}[A-Z])", # Not found in this document
        "mobile": r"Mobile:\s*([0-9]{10})", # Not found in this document
        "email": r"Email:\s*([\w\.-]+@[\w\.-]+)", # Not found in this document
        "statement_period": r"Account Statement from (\d{2}-\d{2}-\d{4}) to (\d{2}-\d{2}-\d{4})" # Adjusted pattern
    }

    for key, pattern in patterns.items():
        # Skip ifsc and account_number as they are handled above
        if key in ["account_number", "ifsc"]:
            continue

        # Use re.DOTALL to allow '.' to match newline characters for multi-line patterns
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if match:
            if key == "statement_period":
                data[key] = f"{match.group(1)} to {match.group(2)}".strip()
            else:
                data[key] = match.group(1).strip() if len(match.groups()) >= 1 else match.group(0).strip()
        else:
            data[key] = None
    return data

def extract_transactions(text: str) -> List[Dict[str, Any]]:
    """Parses tabular transaction lines into clean, structured dictionaries."""
    # Updated pattern: description and optional cheque_ref are now captured in a single group (Group 2)
    # The pattern relies on the consistent format of the last three fields: Debit, Credit, and Balance.
    transaction_pattern = re.compile(
        r"^\s*\d+\s+"                                    # Sr.No (not captured as group)
        r"(\d{2}-\d{2}-\d{4})\s+"                       # Group 1: Transaction Date (date_str)
        r"(?:\d{2}-\d{2}-\d{4}|\s+)?\s*"               # Value Date (optional, non-capturing) with optional space
        r"(.+?)"                                        # Group 2: Description (greedy for anything, but stops before the amounts)
        r"\s+([\d.,]+|\.)"                              # Group 3: Debit amount
        r"\s+([\d.,]+|\.)"                              # Group 4: Credit amount
        r"\s+([\d.,]+)",                                # Group 5: Balance
        re.MULTILINE
    )

    # Pattern for opening balance lines, including Sr.No and Date
    opening_balance_pattern = re.compile(
        r"^\s*\d+\s+(\d{2}-\d{2}-\d{4})\s+Opening Balance\s+-\s+-\s+([\d.,]+)",
        re.IGNORECASE
    )

    transactions = []

    lines = text.split('\n')
    for line in lines:
        # Try to match opening balance first
        ob_match = opening_balance_pattern.search(line)
        if ob_match:
            date_str, balance_str = ob_match.groups()
            transactions.append({
                "date": date_str.strip(),
                "description": "Opening Balance",
                "transaction_type": "UNKNOWN", # Or 'BALANCE_ADJUSTMENT'
                "amount": 0.0,
                "balance": float(balance_str.replace(',', ''))
            })
            continue # Move to next line

        # Then try to match general transaction pattern
        match = transaction_pattern.search(line)
        if match:
            # Corrected group indexing based on the new regex capturing groups
            # Group 1: date_str, Group 2: description, Group 3: debit_str, Group 4: credit_str, Group 5: balance_str
            date_str, description, debit_str, credit_str, balance_str = match.groups()

            # Clean and convert amounts
            debit = float(debit_str.replace(',', '')) if debit_str and debit_str.strip() != '.' else 0.0
            credit = float(credit_str.replace(',', '')) if credit_str and credit_str.strip() != '.' else 0.0
            # Fix: Handle '.' for balance_str similar to debit/credit
            balance = float(balance_str.replace(',', '')) if balance_str and balance_str.strip() != '.' else 0.0

            # Determine transaction type and amount
            transaction_type = "DEBIT" if debit > 0 else ("CREDIT" if credit > 0 else "UNKNOWN")
            amount = debit if debit > 0 else credit

            # Description now includes cheque_ref if it was present in the raw text
            full_description = description.strip()

            transactions.append({
                "date": date_str.strip(),
                "description": full_description.strip(),
                "transaction_type": transaction_type,
                "amount": amount,
                "balance": balance
            })
    return transactions

print("✅ Data Extraction Layer loaded.")

✅ Data Extraction Layer loaded.


Cell 5: Pydantic Data Integrity Schema Rules
This enforces data compliance and stops contaminated or mis-parsed text from reaching down-stream systems.

In [ ]:
class Transaction(BaseModel):
    date: str
    description: str
    transaction_type: str
    amount: float
    balance: float
    category: Optional[str] = None
    sentiment: Optional[str] = None # Added new field for sentiment analysis

    @field_validator("transaction_type")
    @classmethod
    def valid_txn_type(cls, value: str) -> str:
        if value not in ["DEBIT", "CREDIT", "UNKNOWN"]:
            raise ValueError("transaction_type must be DEBIT, CREDIT or UNKNOWN")
        return value

class BankDocument(BaseModel):
    bank_name: str
    customer_name: str
    account_number: Optional[str] = None # Made optional
    ifsc: Optional[str] = None # Made optional
    pan: Optional[str] = None # Made optional
    mobile: Optional[str] = None # Made optional
    email: Optional[str] = None # Made optional
    statement_period: str
    transactions: List[Transaction]
    extraction_metadata: Dict[str, Any]

    @field_validator("pan")
    @classmethod
    def validate_pan(cls, value: Optional[str]) -> Optional[str]:
        if value is None:
            return value # Allow None if the field is optional
        if not re.match(r"^[A-Z]{5}[0-9]{4}[A-Z]$", value):
            raise ValueError("Invalid PAN format")
        return value

    @field_validator("ifsc")
    @classmethod
    def validate_ifsc(cls, value: Optional[str]) -> Optional[str]:
        if value is None:
            return value # Allow None if the field is optional
        # Updated regex to accept '0' or 'O' in the fifth position for IFSC
        if not re.match(r"^[A-Z]{4}[0O][A-Z0-9]{6}$", value):
            raise ValueError("Invalid IFSC format")
        return value

    @field_validator("account_number")
    @classmethod
    def validate_account_number(cls, value: Optional[str]) -> Optional[str]:
        if value is None:
            return value # Allow None if the field is optional
        # Update regex to be more flexible, as account numbers can vary widely
        if not re.match(r"^[A-Z0-9\s]{5,20}$", value): # Allows alphanumeric and spaces, 5-20 chars
            raise ValueError("Invalid account number format")
        return value

print("✅ Validation Schema compiled.")

✅ Validation Schema compiled.


Cell 6: Business Intelligence & Risk Architecture Layer
This module adds financial logic by scanning transaction lines to tag warning flags.

Python

In [ ]:
def categorize_transaction(description: str) -> str:
    desc = description.upper()
    # More specific rules first
    if "SALARY" in desc or "PAYCHECK" in desc or "WAGES" in desc:
        return "Income"
    elif "SWIGGY" in desc or "ZOMATO" in desc or "DOMINOES" in desc or "UBER EATS" in desc or "RESTAURANT" in desc or "CAFE" in desc:
        return "Food & Dining"
    elif "ATM CASH" in desc or "CASH WITHDRAWAL" in desc:
        return "Cash Withdrawal"
    elif "UPI" in desc or "NEFT" in desc or "IMPS" in desc or "TRANSFER" in desc:
        return "Transfer"
    elif "RENT" in desc or "HOUSING" in desc or "MORTGAGE" in desc:
        return "Housing"
    elif "ELECTRICITY" in desc or "WATER BILL" in desc or "GAS BILL" in desc:
        return "Utilities"
    elif "LOAN" in desc or "EMI" in desc or "INSTALLMENT" in desc:
        return "Loan Payment"
    elif "INSURANCE" in desc or "PREMIUM" in desc:
        return "Insurance"
    elif "GROCERY" in desc or "SUPERMARKET" in desc or "BIGBAZAR" in desc or "DMART" in desc:
        return "Groceries"
    elif "FUEL" in desc or "PETROL" in desc or "GAS STATION" in desc:
        return "Transportation"
    elif "AMAZON" in desc or "FLIPKART" in desc or "MYNTRA" in desc or "SHOPPING" in desc:
        return "Shopping"
    elif "NETFLIX" in desc or "SPOTIFY" in desc or "YOUTUBE PREMIUM" in desc or "SUBSCRIPTION" in desc:
        return "Subscription"
    elif "MEDICAL" in desc or "PHARMACY" in desc or "HOSPITAL" in desc:
        return "Healthcare"
    elif "INVESTMENT" in desc or "MUTUAL FUND" in desc or "STOCKS" in desc:
        return "Investments"
    else:
        return "Miscellaneous"

def analyze_sentiment(description: str) -> str:
    desc = description.upper()
    if "REFUND" in desc or "REVERSAL" in desc or "ADJUSTMENT" in desc:
        return "Positive"
    elif "FRAUD" in desc or "UNAUTHORIZED" in desc or "DISPUTE" in desc or "OVERDUE" in desc:
        return "Negative"
    else:
        return "Neutral"

def generate_risk_flags(doc: Dict[str, Any]) -> List[Dict[str, Any]]:
    flags = []
    for txn in doc["transactions"]:
        desc = txn["description"].upper()
        amount = txn["amount"]

        if txn["transaction_type"] == "DEBIT" and amount >= 50000:
            flags.append({"severity": "HIGH", "rule": "HIGH_VALUE_DEBIT", "message": f"High-value debit of {amount} detected", "transaction": txn})
        if txn["transaction_type"] == "CREDIT" and amount >= 100000:
            flags.append({"severity": "MEDIUM", "rule": "HIGH_VALUE_CREDIT", "message": f"High-value credit of {amount} detected", "transaction": txn})
        if "UNKNOWN" in desc:
            flags.append({"severity": "HIGH", "rule": "UNKNOWN_BENEFICIARY", "message": "Transfer to unknown beneficiary detected", "transaction": txn})
        if "INTERNATIONAL" in desc:
            flags.append({"severity": "HIGH", "rule": "INTERNATIONAL_TRANSFER", "message": "International transfer detected", "transaction": txn})
        if "CASH DEPOSIT" in desc and amount % 10000 == 0:
            flags.append({"severity": "MEDIUM", "rule": "ROUND_CASH_DEPOSIT", "message": "Round-number cash deposit detected", "transaction": txn})
    return flags

def ai_summary_pipeline(doc: Dict[str, Any], flags: List[Dict[str, Any]]) -> Dict[str, Any]:
    high_flags = [f for f in flags if f["severity"] == "HIGH"]
    medium_flags = [f for f in flags if f["severity"] == "MEDIUM"]
    decision = "MANUAL_REVIEW_REQUIRED" if len(high_flags) >= 2 else ("REVIEW_RECOMMENDED" if len(high_flags) == 1 else "LOW_RISK")
    risk_score = 85 if len(high_flags) >= 2 else (65 if len(high_flags) == 1 else 30)

    return {
        "customer_name": doc["customer_name"],
        "bank_name": doc["bank_name"],
        "total_transactions": len(doc["transactions"]),
        "high_risk_flags": len(high_flags),
        "medium_risk_flags": len(medium_flags),
        "risk_score": risk_score,
        "recommended_action": decision,
        "executive_summary": f"{doc['customer_name']} has {len(doc['transactions'])} items. System tracked {len(high_flags)} high-risk anomalies. Action: {decision}."
    }

print("✅ Risk & Analytics Modules loaded.")

✅ Risk & Analytics Modules loaded.


Cell 7: Unified Orchestration & Audit Engine
This stitches all the steps together, creating immutable logs for verification.

In [ ]:
audit_logs = []

def log_event(stage: str, status: str, details: Dict[str, Any]):
    # Uses Pydantic v2 safe timezone-aware UTC implementation
    event = {
        "event_id": str(uuid.uuid4()),
        "stage": stage,
        "status": status,
        "timestamp": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "details": details
    }
    audit_logs.append(event)
    print(f"[Workflow Progress] Stage: {stage}, Status: {status}, Details: {details.get('message', '') or details.get('recommended_action', '') or details.get('document_path', '')}")

def enterprise_document_ai_workflow(document_path: str) -> Dict[str, Any]:
    global audit_logs
    audit_logs = [] # Reset trace logs for each run

    log_event("DOCUMENT_RECEIVED", "SUCCESS", {"document_path": document_path})
    # Prepend /content/ to the document_path to ensure correct file access
    full_document_path = os.path.join("/content/", document_path)
    text = ocr_engine.extract_text(full_document_path)

    entities = extract_header_entities(text)
    txns = extract_transactions(text)

    # Categorize and analyze sentiment for transactions
    for txn in txns:
        txn["category"] = categorize_transaction(txn["description"])
        txn["sentiment"] = analyze_sentiment(txn["description"])

    log_event("AI_UNDERSTANDING", "SUCCESS", {"entities_found": list(entities.keys()), "transactions": len(txns)})

    doc = {
        **entities,
        "transactions": txns,
        "extraction_metadata": {
            "document_id": str(uuid.uuid4()),
            "ocr_provider": "SimulatedOCRProvider",
            "extracted_at": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
            "confidence_score": 0.94
        }
    }

    try:
        BankDocument(**doc)
        log_event("VALIDATION", "SUCCESS", {"message": "Schema validation passed"})
    except ValidationError as e:
        log_event("VALIDATION", "FAILED", {"errors": str(e)})
        return {"status": "FAILED", "errors": str(e), "audit_logs": audit_logs}

    flags = generate_risk_flags(doc)
    summary = ai_summary_pipeline(doc, flags)
    log_event("SUMMARY_GENERATION", "SUCCESS", {"recommended_action": summary["recommended_action"]})

    return {
        "status": "SUCCESS",
        "document": doc,
        "risk_flags": flags,
        "risk_summary": summary,
        "audit_logs": audit_logs
    }

print("✅ Orchestration Workflow Ready.")

✅ Orchestration Workflow Ready.


Cell 8: Asynchronous API Server Deployment
This runs a FastAPI gateway in a separate thread, letting students query their engine live without freezing the notebook execution ce

### Why FastAPI for this workflow?
FastAPI is used here to deploy the `enterprise_document_ai_workflow` as a robust and scalable web API. This approach offers several advantages:

1.  **External Integration**: It allows any external application (e.g., a web frontend, a mobile app, or another backend service, as simulated in Cell 9) to easily send document requests to our AI engine and receive structured results.
2.  **Modularity and Decoupling**: By wrapping the workflow in an API, the AI logic becomes a decoupled microservice. This means the core AI engine can be developed and updated independently from the applications that consume it.
3.  **Real-time Processing**: It enables real-time document processing, where documents can be submitted and analyzed immediately, with results returned as soon as they are available.
4.  **Standardized Interface**: FastAPI automatically generates interactive API documentation (Swagger UI/OpenAPI), providing a clear and standardized way for developers to understand and interact with the service.
5.  **Performance**: FastAPI is known for its high performance, which is critical for potentially high-volume document processing tasks.

In essence, FastAPI transforms our analytical Python code into a production-ready service that can be easily consumed by other systems.

In [ ]:
app = FastAPI(title="GenAI Pioneer Enterprise Document AI API")

class DocumentRequest(BaseModel):
    document_path: str

@app.post("/process-document")
def process_document(request: DocumentRequest):
    # Ensure the workflow uses the global ocr_engine and other updated components
    # By default, functions defined at module level capture globals at definition time.
    # Rerunning this cell re-defines the app and its routes, picking up new globals.
    result = enterprise_document_ai_workflow(request.document_path)
    if result["status"] == "FAILED":
        raise HTTPException(status_code=422, detail=result["errors"])
    return {
        "status": result["status"],
        "risk_summary": result.get("risk_summary"),
        "risk_flags_count": len(result.get("risk_flags", [])),
        "audit_log_count": len(result.get("audit_logs", []))
    }

# Global variable to hold the server instance
server = None
server_thread = None

def run_server():
    global server # Declare intent to modify the global server variable
    global server_thread

    if server_thread and server_thread.is_alive():
        print("Server already running. Attempting to stop...")
        # In a real application, you'd have a clean shutdown hook.
        # In Colab, often the kernel restart is the most reliable way.
        # For this context, we'll just try to stop uvicorn's internal server if possible
        # or rely on port binding error if a new server is started.
        try:
            if server:
                server.should_exit = True
                # server_thread.join(timeout=5) # Wait for thread to finish if it has clean exit logic
                print("Attempted to signal server to stop.")
        except Exception as e:
            print(f"Error stopping previous server: {e}")

    nest_asyncio.apply()
    config = uvicorn.Config(app, host="127.0.0.1", port=9000, log_level="info")
    server = uvicorn.Server(config=config)
    print("Starting new server instance...")
    server.run()

# Ensure only one server thread is started at a time
if server_thread is None or not server_thread.is_alive():
    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()
    print("🚀 Backend REST API running smoothly on network framework address: http://127.0.0.1:9000")
else:
    print("Server thread already active. No new server started. If you made changes, consider restarting the kernel or re-running this cell after stopping the previous server.")

Server already running. Attempting to stop...🚀 Backend REST API running smoothly on network framework address: http://127.0.0.1:9000



Cell 9: Live Integration Test (The Verification Step)
This acts as a mock downstream application (like a Java/Spring Boot server) calling the Python AI engine.

In [ ]:
# Simulate an external server calling our live endpoint
payload = {"document_path": "BANK STATEMENT 2.pdf"}
response = requests.post("http://127.0.0.1:9000/process-document", json=payload)

print("--- DOWNSTREAM SERVER INGESTION RESULTS ---")
print(json.dumps(response.json(), indent=2))

[Workflow Progress] Stage: DOCUMENT_RECEIVED, Status: SUCCESS, Details: BANK STATEMENT 2.pdf
Successfully extracted text from /content/BANK STATEMENT 2.pdf.
[Workflow Progress] Stage: AI_UNDERSTANDING, Status: SUCCESS, Details: 
[Workflow Progress] Stage: VALIDATION, Status: SUCCESS, Details: Schema validation passed
[Workflow Progress] Stage: SUMMARY_GENERATION, Status: SUCCESS, Details: REVIEW_RECOMMENDED
INFO:     127.0.0.1:49988 - "POST /process-document HTTP/1.1" 200 OK
--- DOWNSTREAM SERVER INGESTION RESULTS ---
{
  "status": "SUCCESS",
  "risk_summary": {
    "customer_name": "CH JANAKI RAGHU RAMI REDDY MADHAVANAGAR KURNOOL",
    "bank_name": "World Bank of Baroda",
    "total_transactions": 24,
    "high_risk_flags": 1,
    "medium_risk_flags": 0,
    "risk_score": 65,
    "recommended_action": "REVIEW_RECOMMENDED",
    "executive_summary": "CH JANAKI RAGHU RAMI REDDY MADHAVANAGAR KURNOOL has 24 items. System tracked 1 high-risk anomalies. Action: REVIEW_RECOMMENDED."
  },
  "r

In [ ]:
# Simulate an external server calling our live endpoint
payload = {"document_path": "/content/BANK STATEMENT 2.pdf"} # Ensure full path for consistency
response = requests.post("http://127.0.0.1:9000/process-document", json=payload)

print("-- DOWNSTREAM SERVER INGESTION RESULTS ---")
print(json.dumps(response.json(), indent=2))

[Workflow Progress] Stage: DOCUMENT_RECEIVED, Status: SUCCESS, Details: /content/BANK STATEMENT 2.pdf
Successfully extracted text from /content/BANK STATEMENT 2.pdf.
[Workflow Progress] Stage: AI_UNDERSTANDING, Status: SUCCESS, Details: 
[Workflow Progress] Stage: VALIDATION, Status: SUCCESS, Details: Schema validation passed
[Workflow Progress] Stage: SUMMARY_GENERATION, Status: SUCCESS, Details: REVIEW_RECOMMENDED
INFO:     127.0.0.1:49288 - "POST /process-document HTTP/1.1" 200 OK
-- DOWNSTREAM SERVER INGESTION RESULTS ---
{
  "status": "SUCCESS",
  "risk_summary": {
    "customer_name": "CH JANAKI RAGHU RAMI REDDY MADHAVANAGAR KURNOOL",
    "bank_name": "World Bank of Baroda",
    "total_transactions": 24,
    "high_risk_flags": 1,
    "medium_risk_flags": 0,
    "risk_score": 65,
    "recommended_action": "REVIEW_RECOMMENDED",
    "executive_summary": "CH JANAKI RAGHU RAMI REDDY MADHAVANAGAR KURNOOL has 24 items. System tracked 1 high-risk anomalies. Action: REVIEW_RECOMMENDED."
 

### Direct Workflow Execution Demo
To explicitly see the `[Workflow Progress]` messages generated by the `log_event` function, we can call the `enterprise_document_ai_workflow` function directly in a cell.

In [ ]:
print("\n--- DIRECT WORKFLOW EXECUTION --- ")
direct_result = enterprise_document_ai_workflow("BANK STATEMENT 2.pdf")
print("\n--- DIRECT WORKFLOW RESULT ---")

# Handle cases where direct_result might be a 'FAILED' dictionary
if direct_result["status"] == "SUCCESS":
    print(json.dumps({
        "status": direct_result["status"],
        "risk_summary": direct_result["risk_summary"],
        "risk_flags_count": len(direct_result["risk_flags"]),
        "audit_log_count": len(direct_result["audit_logs"])
    }, indent=2))
else:
    print(json.dumps({
        "status": direct_result["status"],
        "errors": direct_result.get("errors", "Unknown error occurred during workflow execution."),
        "audit_log_count": len(direct_result["audit_logs"])
    }, indent=2))


--- DIRECT WORKFLOW EXECUTION --- 
[Workflow Progress] Stage: DOCUMENT_RECEIVED, Status: SUCCESS, Details: BANK STATEMENT 2.pdf
Successfully extracted text from /content/BANK STATEMENT 2.pdf.
[Workflow Progress] Stage: AI_UNDERSTANDING, Status: SUCCESS, Details: 
[Workflow Progress] Stage: VALIDATION, Status: SUCCESS, Details: Schema validation passed
[Workflow Progress] Stage: SUMMARY_GENERATION, Status: SUCCESS, Details: REVIEW_RECOMMENDED

--- DIRECT WORKFLOW RESULT ---
{
  "status": "SUCCESS",
  "risk_summary": {
    "customer_name": "CH JANAKI RAGHU RAMI REDDY MADHAVANAGAR KURNOOL",
    "bank_name": "World Bank of Baroda",
    "total_transactions": 24,
    "high_risk_flags": 1,
    "medium_risk_flags": 0,
    "risk_score": 65,
    "recommended_action": "REVIEW_RECOMMENDED",
    "executive_summary": "CH JANAKI RAGHU RAMI REDDY MADHAVANAGAR KURNOOL has 24 items. System tracked 1 high-risk anomalies. Action: REVIEW_RECOMMENDED."
  },
  "risk_flags_count": 1,
  "audit_log_count": 4
}

In [ ]:
# Demonstrate categorized transactions with refined rules
print("\n--- CATEGORIZED TRANSACTIONS (REFINED RULES) ---")
categorized_result_refined = enterprise_document_ai_workflow("BANK STATEMENT 2.pdf")

if categorized_result_refined["status"] == "SUCCESS":
    for i, txn in enumerate(categorized_result_refined["document"]["transactions"]):
        print(f"Transaction {i+1}: Description: {txn['description']}, Amount: {txn['amount']}, Category: {txn['category']}")
else:
    print("Workflow failed during refined categorization demonstration.")


--- CATEGORIZED TRANSACTIONS (REFINED RULES) ---
[Workflow Progress] Stage: DOCUMENT_RECEIVED, Status: SUCCESS, Details: BANK STATEMENT 2.pdf
Successfully extracted text from /content/BANK STATEMENT 2.pdf.
[Workflow Progress] Stage: AI_UNDERSTANDING, Status: SUCCESS, Details: 
[Workflow Progress] Stage: VALIDATION, Status: SUCCESS, Details: Schema validation passed
[Workflow Progress] Stage: SUMMARY_GENERATION, Status: SUCCESS, Details: REVIEW_RECOMMENDED
Transaction 1: Description: Opening Balance, Amount: 0.0, Category: Miscellaneous
Transaction 2: Description: isaac ceca aeaaaad, Amount: 17000.0, Category: Miscellaneous
Transaction 3: Description: asa 7289/28-41:06/UP ichitreddynymavathi, Amount: 17000.0, Category: Miscellaneous
Transaction 4: Description: pr a TOSOTOSESSIA ASS: O/UPISSOBEBEBOS IDFA, Amount: 1200.0, Category: Transfer
Transaction 5: Description: pp /470608513469117:54:34/UPlIq069221199@ybiN, Amount: 100.0, Category: Miscellaneous
Transaction 6: Description: UR SEER

In [ ]:
# Demonstrate sentiment analysis for transactions
print("\n--- TRANSACTIONS WITH SENTIMENT ANALYSIS ---")
sentiment_result = enterprise_document_ai_workflow("BANK STATEMENT 2.pdf") # Changed filename to an existing one

if sentiment_result["status"] == "SUCCESS":
    for i, txn in enumerate(sentiment_result["document"]["transactions"]):
        print(f"Transaction {i+1}: Description: {txn['description']}, Category: {txn['category']}, Sentiment: {txn['sentiment']}")
else:
    print("Workflow failed during sentiment analysis demonstration.")


--- TRANSACTIONS WITH SENTIMENT ANALYSIS ---
[Workflow Progress] Stage: DOCUMENT_RECEIVED, Status: SUCCESS, Details: BANK STATEMENT 2.pdf
Successfully extracted text from /content/BANK STATEMENT 2.pdf.
[Workflow Progress] Stage: AI_UNDERSTANDING, Status: SUCCESS, Details: 
[Workflow Progress] Stage: VALIDATION, Status: SUCCESS, Details: Schema validation passed
[Workflow Progress] Stage: SUMMARY_GENERATION, Status: SUCCESS, Details: REVIEW_RECOMMENDED
Transaction 1: Description: Opening Balance, Category: Miscellaneous, Sentiment: Neutral
Transaction 2: Description: isaac ceca aeaaaad, Category: Miscellaneous, Sentiment: Neutral
Transaction 3: Description: asa 7289/28-41:06/UP ichitreddynymavathi, Category: Miscellaneous, Sentiment: Neutral
Transaction 4: Description: pr a TOSOTOSESSIA ASS: O/UPISSOBEBEBOS IDFA, Category: Transfer, Sentiment: Neutral
Transaction 5: Description: pp /470608513469117:54:34/UPlIq069221199@ybiN, Category: Miscellaneous, Sentiment: Neutral
Transaction 6: De